<br><h3 style='text-align: center'>Logistic Regression Using Bag Of Words Model</h3><br>

In [2]:
# libraries for part 1
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
import re
import sys

pd.set_option('display.max_columns', 200)
plt.style.use("ggplot")

# Libraries for part 2
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import scipy.sparse
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

In [3]:
df = pl.read_csv("./data/995,000_rows_preprocessed.csv", n_rows=20_000, ignore_errors=True).to_pandas()

df = df[['type', 'content']].rename(columns={'type':'isfake'})

real_labels = ['reliable', 'political']
drop_labels = ['unknown', 'satire', 'bias', 'clickbait', 'state', 'hate', 'nan', '2018-02-10 13:43:39.521661']

df = df.loc[~df['isfake'].isin(drop_labels)]

df['isfake'] = df['isfake'].map(lambda x : 0 if x in real_labels else 1)


n = df.shape[0]
indices = np.arange(n)
labels = df['isfake'].to_numpy()

# First split: 80 % train, 20 % temp (will become val + test)
train_idx, temp_idx = train_test_split(
    indices, test_size=0.2, random_state=1, stratify=labels
)
# Second split: halve the temp set into 10 % val + 10 % test
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=1, stratify=labels[temp_idx]
)

df_train = df.iloc[train_idx]
df_val = df.iloc[val_idx]
df_test = df.iloc[test_idx]

df_train

,isfake,content
101,1,puerto rico homeown brace anoth disast foreclo...
15820,0,<NUM> gun surrend birmingham gun buyback progr...
2570,0,new program introduc trump administr seem insp...
3745,0,former darpa director regina dugan discuss art...
10672,1,true blue conserv report contributor profil st...
...,...,...
3904,1,las cruce nm new mexico man su citi las cruce ...
12055,1,debt insan anyon washington even care <NUM> tr...
2041,1,maxim person brand job hunt reader think stori...
545,0,gold coast via wyong madeenati craig brennan -...


In [4]:
# Given a pandas or polars dataframe and potentially an number of tokens 
# to be included as features, returns a bag of words representation of
# the input's column 'content' as a scipy.sparse.csr_matrix.
def bow(df: pd.DataFrame | pl.DataFrame,
        vectorizer,
        save_as: None | str = None,
        train=True):

    # Create BoW using the CountVectorizer module
    # Define corpus using polars for efficiency
    if type(df) == pd.DataFrame:
        pl_series = pl.from_pandas(df['content'])
    else:
        pl_series = df.select('content')

    # Concat natural language cols to single list
    docs = pl_series.to_list()

    # Fit vectorizer to create compressed sparse column matrix for RAM efficiency
    if train:
        X = vectorizer.fit_transform(docs)
    else:
        X = vectorizer.transform(docs)

    if save_as: scipy.sparse.save_npz(save_as, X)
    
    return X

In [10]:
# Create BoW representation of trainig, validation and test set

# Initialize vectorizer
vec = CountVectorizer(max_features=int(1e6), ngram_range=(1,3), binary=True)

X_train = bow(df_train, vec, save_as='./data/X_train.npz')
X_val = bow(df_val, vec, save_as='./data/X_val.npz', train=False)
X_test = bow(df_test, vec, save_as='./data/X_test.npz', train=False)

y_train = df_train['isfake']
y_val = df_val['isfake']
y_test = df_test['isfake']

In [11]:
classifier = LogisticRegression(random_state=1000, max_iter=2500)
classifier.fit(X_train, y_train)

preds = classifier.predict(X_val)
print(classification_report(preds, y_val))

              precision    recall  f1-score   support

           0       0.90      0.92      0.91       773
           1       0.91      0.89      0.90       713

    accuracy                           0.90      1486
   macro avg       0.90      0.90      0.90      1486
weighted avg       0.90      0.90      0.90      1486



In [ ]:
# # silly catboost test
# from catboost import CatBoostClassifier

# catboost = CatBoostClassifier(verbose=0, random_state=1000)
# catboost.fit(X_train, y_train)

# preds = catboost.predict(X_val)
# print(classification_report(preds, y_val))

              precision    recall  f1-score   support

           0       0.90      0.91      0.91       782
           1       0.90      0.89      0.89       704

    accuracy                           0.90      1486
   macro avg       0.90      0.90      0.90      1486
weighted avg       0.90      0.90      0.90      1486

